In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_squared_error

# START: Vivian Lin

### Load Dataset

Feature Selection:
- `LivingArea`
- `BathroomsTotalIntege`
- `BedroomsTotal`
- `Bed_Bath_Ratio`
- `Property_Age`
- `Months_From_Dec_2025`
- `Sqft_Per_Bedroom`
- `Lot_Utilization`
- `ZipMedianPrice`: obtained by taking the median of each postal code and assigning it to each row of the dataset by postal code

Dependent Variable: `LogClosePrice`

In [29]:
tree_features = [
    'LivingArea','BathroomsTotalInteger',
    'BedroomsTotal','Bed_Bath_Ratio','Property_Age',
    'Months_From_Dec_2025','Sqft_Per_Bedroom','Lot_Utilization', 'ZipMedianPrice'
]

`X_train`/`X_test`: feature metrics

`y_train`/`y_test`: original target

In [38]:
# Load dataset with feature engineering
train = pd.read_csv('../processed_data/train_cleaned_fe.csv', engine='python', on_bad_lines='skip')
test  = pd.read_csv('../processed_Data/test_cleaned_fe.csv', engine='python', on_bad_lines='skip')


# Replace inf and -inf with NAN
train = train.replace([np.inf, -np.inf], np.nan)

train['LogClosePrice'] = np.log(train['ClosePrice'])
test['LogClosePrice'] = np.log(test['ClosePrice'])

zip_median_price = (
    train
    .groupby('PostalCode')['LogClosePrice']
    .median()
)

train['ZipMedianPrice'] = train['PostalCode'].map(zip_median_price)
test['ZipMedianPrice']  = test['PostalCode'].map(zip_median_price)

#Drop rows with NAN values
train = train.dropna()
test = test.dropna()

# X and y of train set
X_train = train[tree_features]
y_train = train['LogClosePrice']

# X and y of test set
X_test = test[tree_features]
y_test = test['LogClosePrice']

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

#### Fitting without hyperparameter tuning

In [33]:
xg_model = XGBRegressor()
xg_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

#### Evaluating XGBoost Model with Default Hyperparameters

In [34]:
# Model Predictions
no_hyper_pred_train = xg_model.predict(X_train)
no_hyper_pred_test = xg_model.predict(X_test)

# R^2 on a Log Scale
no_hyper_R2_train = r2_score(y_train, no_hyper_pred_train)
print(f'Train Set -- R^2 on Log Scale: {no_hyper_R2_train}')

no_hyper_R2_test = r2_score(y_test, no_hyper_pred_test)
print(f'Test Set -- R^2 on Log Scale: {no_hyper_R2_test}')

# MAPE
no_hyper_mape_train = mean_absolute_percentage_error(y_train, no_hyper_pred_train)
print(f'Train Set -- MAPE: {no_hyper_mape_train*100}%')

no_hyper_mape_test = mean_absolute_percentage_error(y_test, no_hyper_pred_test)
print(f'Test Set -- MAPE: {no_hyper_mape_test*100}%')

#MdAPE
no_hyper_mdape_train = np.median(np.abs((y_train - no_hyper_pred_train) / y_train))
print(f'Train Set -- MdAPE: {no_hyper_mdape_train * 100}%')

no_hyper_mdape_test = np.median(np.abs((y_test - no_hyper_pred_test) / y_test))
print(f'Test Set -- MdAPE: {no_hyper_mdape_test * 100}%')


Train Set -- R^2 on Log Scale: 0.9285795820454588
Test Set -- R^2 on Log Scale: 0.8944507689651088
Train Set -- MAPE: 0.8777726111919034%
Test Set -- MAPE: 1.0308123646244916%
Train Set -- MdAPE: 0.6627043903820253%
Test Set -- MdAPE: 0.7522079198418218%


In [40]:
# Predictions
no_hyper_pred_train = xg_model.predict(X_train)
no_hyper_pred_test  = xg_model.predict(X_test)

# R^2 (log scale)
print(f'Train R2 (log): {r2_score(y_train, no_hyper_pred_train)}')
print(f'Test R2 (log): {r2_score(y_test, no_hyper_pred_test)}')

# Convert to actual prices
y_train_actual = np.exp(y_train)
y_test_actual  = np.exp(y_test)

pred_train_actual = np.exp(no_hyper_pred_train)
pred_test_actual  = np.exp(no_hyper_pred_test)

# R^2 (actual scale)
print(f'Train R2 (actual): {r2_score(y_train_actual, pred_train_actual)}')
print(f'Test R2 (actual): {r2_score(y_test_actual, pred_test_actual)}')

# MAPE
print(f'Train MAPE: {mean_absolute_percentage_error(y_train_actual, pred_train_actual) * 100}%')
print(f'Test MAPE: {mean_absolute_percentage_error(y_test_actual, pred_test_actual) * 100}%')

# MdAPE
print(f'Train MdAPE: {np.median(np.abs((y_train_actual - pred_train_actual) / y_train_actual)) * 100}%')
print(f'Test MdAPE: {np.median(np.abs((y_test_actual - pred_test_actual) / y_test_actual)) * 100}%')

Train R2 (log): 0.9285795820454588
Test R2 (log): 0.8944507689651088
Train R2 (actual): 0.8996725644227653
Test R2 (actual): 0.833195628312196
Train MAPE: 12.280358156012465%
Test MAPE: 14.523243185434143%
Train MdAPE: 9.05621710526324%
Test MdAPE: 10.24824496730346%


### Key Evaluation Metrics

| Metric       | Training Set | Testing Set |
|-------------|-------------|------------|
| **R² (log scale)**      | 0.929     | 0.894      |
| **R² (actual scale)**      | 0.900      | 0.833     |
| **MAPE (%)**| 12.280%       | 14.523%      |
| **MdAPE (%)**| 9.056%      | 10.248%      |

### Interpretation [need to be revised]

1. **High R²**: The model explains ~93% of variance on training data and ~89% on unseen test data, indicating that most of the variation in the target variable cannot be explained by the model.  
2. **Low MAPE and MdAPE**: The mean and median absolute percentage errors are  low, meaning predictions are, on average, within ~9-15% of the true property prices.
3. **Robustness**: The difference between training and testing MAPE, MdAPE, and $R^2$ suggests the model is slightly overfitting and handles unseen data decently well.
4. **Practical Use**: This XGBoost model is effective at producing accurate predictions of residential property closing prices, as indicated by the low MAPE and MdAPE values. However, the relatively low $R^2$ suggests that the model explains only a limited portion of the variation in the target variable LogClosePrice, meaning that important factors influencing property prices may not be captured by the current feature set.

# END: Vivian Lin